(Baseline_Feature_Transformation)=
# Transformação de características de linha de base

O conjunto de dados simulado gerado na seção anterior é simples. Ele contém apenas as características essenciais que caracterizam uma transação de cartão de pagamento. São elas: um identificador único para a transação, a data e hora da transação, o valor da transação, um identificador único para o cliente, um número único para o comerciante e uma variável binária que rotula a transação como legítima ou fraudulenta (0 para legítima ou 1 para fraudulenta). A Fig. 1 mostra as três primeiras linhas do conjunto de dados simulado:
 
![alt text](images/tx_table.png)
<p style="text-align: center;">
Fig. 1. As três primeiras transações no conjunto de dados simulado usado neste capítulo.
</p>

O que cada linha essencialmente resume é que, às 00:00:31, em 1º de abril de 2018, um cliente com ID 596 fez um pagamento de 57,19 a um comerciante com ID 3156, e que a transação não foi fraudulenta. Em seguida, às 00:02:10, em 1º de abril de 2018, um cliente com ID 4961 fez um pagamento de 81,51 a um comerciante com ID 3412, e que a transação não foi fraudulenta. E assim por diante. O conjunto de dados simulado é uma longa lista de tais transações (1,8 milhão no total). A variável `transaction_ID` é um identificador único para cada transação.

Embora conceitualmente simples para um humano, tal conjunto de características não é adequado para um modelo preditivo de aprendizado de máquina. Os algoritmos de aprendizado de máquina geralmente requerem características *numéricas* e *ordenadas*. Numéricas significa que o tipo da variável deve ser um inteiro ou um número real. Ordenadas significa que a ordem dos valores de uma variável é significativa.

Neste conjunto de dados, as únicas características numéricas e ordenadas são o valor da transação e o rótulo de fraude. A data é um timestamp do Pandas e, portanto, não é numérica. Os identificadores para transações, clientes e terminais são numéricos, mas não ordenados: não faria sentido assumir, por exemplo, que o terminal com ID 3548 é "maior" ou "superior" ao terminal com ID 1983. Em vez disso, esses identificadores representam "entidades" distintas, que são referidas como características *categóricas*.

Infelizmente, não existe um procedimento padrão para lidar com características não numéricas ou categóricas. O tema é conhecido na literatura de aprendizado de máquina como *engenharia de características* ou *transformação de características*. Em essência, o objetivo da engenharia de características é projetar novas características que se assumem relevantes para um problema preditivo. O design dessas características geralmente depende do problema e envolve conhecimento de domínio.

Nesta seção, implementaremos três tipos de transformação de características que são conhecidos por serem relevantes para a detecção de fraude em cartões de pagamento.

![encoding](images/encoding_variables.png)

O primeiro tipo de transformação envolve a variável de data/hora e consiste em criar características binárias que caracterizam períodos potencialmente relevantes. Criaremos duas dessas características. A primeira caracterizará se uma transação ocorre durante um dia de semana ou durante o fim de semana. A segunda caracterizará se uma transação ocorre durante o dia ou à noite. Essas características podem ser úteis, pois foi observado em conjuntos de dados do mundo real que os padrões fraudulentos diferem entre dias de semana e fins de semana, e entre dia e noite.

O segundo tipo de transformação envolve o ID do cliente e consiste em criar características que caracterizam os comportamentos de gastos do cliente. Seguiremos o framework RFM (Recência, Frequência, Valor Monetário) proposto em {cite}`VANVLASSELAER201538`, e rastrearemos o valor médio de gastos e o número de transações para cada cliente e para três tamanhos de janela temporal. Isso levará à criação de seis novas características.

O terceiro tipo de transformação envolve o ID do terminal e consiste em criar novas características que caracterizam o "risco" associado ao terminal. O risco será definido como o número médio de fraudes observadas no terminal para três tamanhos de janela temporal. Isso levará à criação de três novas características.

A tabela abaixo resume os tipos de transformação que serão realizados e as novas características que serão criadas.

|Nome da característica original|Tipo da característica original|Transformação|Número de novas características|Tipo da(s) nova(s) característica(s)|
|---|---|---|---|---|
|TX\_DATE\_TIME | Timestamp do Pandas |0 se a transação ocorrer em um dia de semana, 1 se ocorrer em um fim de semana. A nova característica é chamada TX_DURING_WEEKEND.|1|Inteiro (0/1)|
|TX\_DATE\_TIME | Timestamp do Pandas |0 se a transação ocorrer entre 6h e 0h, 1 se ocorrer entre 0h e 6h. A nova característica é chamada TX_DURING_NIGHT.|1|Inteiro (0/1)|
|CUSTOMER\_ID | Variável categórica |Número de transações do cliente nos últimos n dia(s), para n em {1,7,30}. As novas características são chamadas CUSTOMER_ID_NB_TX_nDAY_WINDOW.|3|Inteiro|
|CUSTOMER\_ID | Variável categórica |Valor médio de gastos nos últimos n dia(s), para n em {1,7,30}. As novas características são chamadas CUSTOMER_ID_AVG_AMOUNT_nDAY_WINDOW.|3|Real|
|TERMINAL\_ID | Variável categórica |Número de transações no terminal nos últimos n+d dia(s), para n em {1,7,30} e d=7. O parâmetro d é chamado de atraso e será discutido mais adiante neste notebook. As novas características são chamadas TERMINAL_ID_NB_TX_nDAY_WINDOW.|3|Inteiro|
|TERMINAL\_ID | Variável categórica |Número médio de fraudes no terminal nos últimos n+d dia(s), para n em {1,7,30} e d=7. O parâmetro d é chamado de atraso e será discutido mais adiante neste notebook. As novas características são chamadas TERMINAL_ID_RISK_nDAY_WINDOW.|3|Real|

As seções seguintes fornecem a implementação de cada uma dessas três transformações. Após as transformações, um conjunto de 14 novas características será criado. Note que algumas características são resultado de funções de agregação sobre os valores de outras características ou condições (mesmo cliente, janela temporal fornecida). Essas características são frequentemente referidas como *características agregadas*.

In [24]:
# Initialization: Load shared functions and simulated data 

# Load shared functions
!curl -O https://raw.githubusercontent.com/Fraud-Detection-Handbook/fraud-detection-handbook/main/Chapter_References/shared_functions.py
%run shared_functions.py

# Get simulated data from Github repository
if not os.path.exists("simulated-data-raw"):
    !git clone https://github.com/Fraud-Detection-Handbook/simulated-data-raw
        

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31567  100 31567    0     0   135k      0 --:--:-- --:--:-- --:--:--  135k
Cloning into 'simulated-data-raw'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (187/187), done.
remote: Total 189 (delta 0), reused 186 (delta 0), pack-reused 0
Receiving objects: 100% (189/189), 28.04 MiB | 3.13 MiB/s, done.


## Carregamento do conjunto de dados

Vamos primeiro carregar os dados de transação simulados no notebook anterior. Carregaremos os arquivos de transação de abril a setembro. Os arquivos podem ser carregados usando a função `read_from_files` no notebook de [funções compartilhadas](shared_functions). A função foi colocada neste notebook pois será usada frequentemente ao longo deste livro.

A função recebe como entrada a pasta onde os arquivos de dados estão localizados e as datas que definem o período a ser carregado (entre `BEGIN_DATE` e `END_DATE`). Ela retorna um DataFrame de transações. As transações são ordenadas cronologicamente.

In [3]:
DIR_INPUT='./simulated-data-raw/data/' 

BEGIN_DATE = "2018-04-01"
END_DATE = "2018-09-30"

print("Load  files")
%time transactions_df=read_from_files(DIR_INPUT, BEGIN_DATE, END_DATE)
print("{0} transactions loaded, containing {1} fraudulent transactions".format(len(transactions_df),transactions_df.TX_FRAUD.sum()))


Load  files
CPU times: user 3.1 s, sys: 696 ms, total: 3.79 s
Wall time: 4.13 s
1754155 transactions loaded, containing 14681 fraudulent transactions


In [4]:
transactions_df.head()

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0


## Transformações de data e hora

Criaremos duas novas características binárias a partir das datas e horas das transações:

* A primeira caracterizará se uma transação ocorre durante um dia de semana (valor 0) ou um fim de semana (1), e será chamada `TX_DURING_WEEKEND`
* A segunda caracterizará se uma transação ocorre durante o dia (0) ou durante a noite (1). A noite é definida como as horas entre 0h e 6h. Será chamada `TX_DURING_NIGHT`.

Para a característica `TX_DURING_WEEKEND`, definimos uma função `is_weekend` que recebe como entrada um timestamp do Pandas e retorna 1 se a data for durante um fim de semana, ou 0 caso contrário. O objeto timestamp convenientemente fornece a função `weekday` para ajudar no cálculo desse valor.

In [5]:
def is_weekend(tx_datetime):
    
    # Transform date into weekday (0 is Monday, 6 is Sunday)
    weekday = tx_datetime.weekday()
    # Binary value: 0 if weekday, 1 if weekend
    is_weekend = weekday>=5
    
    return int(is_weekend)


É então simples calcular essa característica para todas as transações usando a função `apply` do Pandas.

In [6]:
%time transactions_df['TX_DURING_WEEKEND']=transactions_df.TX_DATETIME.apply(is_weekend)

CPU times: user 7.54 s, sys: 247 ms, total: 7.79 s
Wall time: 7.94 s


Seguimos a mesma lógica para implementar a característica `TX_DURING_NIGHT`. Primeiro, uma função `is_night` que recebe como entrada um timestamp do Pandas e retorna 1 se o horário for noturno, ou 0 caso contrário. O objeto timestamp convenientemente fornece a propriedade `hour` para ajudar no cálculo desse valor.

In [7]:
def is_night(tx_datetime):
    
    # Get the hour of the transaction
    tx_hour = tx_datetime.hour
    # Binary value: 1 if hour less than 6, and 0 otherwise
    is_night = tx_hour<=6
    
    return int(is_night)

In [8]:
%time transactions_df['TX_DURING_NIGHT']=transactions_df.TX_DATETIME.apply(is_night)

CPU times: user 7.09 s, sys: 221 ms, total: 7.31 s
Wall time: 7.47 s


Vamos verificar se essas características foram calculadas corretamente.

In [9]:
transactions_df[transactions_df.TX_TIME_DAYS>=30]

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT
288062,288062,2018-05-01 00:01:21,3546,2944,18.71,2592081,30,0,0,0,1
288063,288063,2018-05-01 00:01:48,206,3521,18.60,2592108,30,0,0,0,1
288064,288064,2018-05-01 00:02:22,2610,4470,66.67,2592142,30,0,0,0,1
288065,288065,2018-05-01 00:03:15,4578,1520,79.41,2592195,30,0,0,0,1
288066,288066,2018-05-01 00:03:51,1246,7809,52.08,2592231,30,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
1754150,1754150,2018-09-30 23:56:36,161,655,54.24,15810996,182,0,0,1,0
1754151,1754151,2018-09-30 23:57:38,4342,6181,1.23,15811058,182,0,0,1,0
1754152,1754152,2018-09-30 23:58:21,618,1502,6.62,15811101,182,0,0,1,0
1754153,1754153,2018-09-30 23:59:52,4056,3067,55.40,15811192,182,0,0,1,0


O dia 2018-05-01 foi uma segunda-feira, e o 2018-09-30 foi um domingo. Essas datas estão corretamente sinalizadas como dia de semana e fim de semana, respectivamente. A característica de dia e noite também está corretamente definida para as primeiras transações, que ocorrem logo após 0h, e as últimas transações que ocorrem logo antes de 0h.

## Transformações do ID do cliente

Vamos agora prosseguir com as transformações do ID do cliente. Nos inspiraremos no framework RFM (Recência, Frequência, Valor Monetário) proposto em {cite}`VANVLASSELAER201538`, e calcularemos duas dessas características em três janelas temporais. A primeira característica será o número de transações que ocorrem dentro de uma janela temporal (Frequência). A segunda será o valor médio gasto nessas transações (Valor Monetário). As janelas temporais serão definidas em um, sete e trinta dias. Isso gerará seis novas características. Note que essas janelas temporais poderiam ser otimizadas posteriormente junto com os modelos usando um procedimento de seleção de modelos ([Capítulo 5](Model_Selection)).

Vamos implementar essas transformações escrevendo uma função `get_customer_spending_behaviour_features`. A função recebe como entradas o conjunto de transações de um cliente e um conjunto de tamanhos de janela. Ela retorna um DataFrame com as seis novas características. Nossa implementação baseia-se na função `rolling` do Pandas, que facilita o cálculo de agregados em uma janela temporal.

In [10]:
def get_customer_spending_behaviour_features(customer_transactions, windows_size_in_days=[1,7,30]):
    
    # Let us first order transactions chronologically
    customer_transactions=customer_transactions.sort_values('TX_DATETIME')
    
    # The transaction date and time is set as the index, which will allow the use of the rolling function 
    customer_transactions.index=customer_transactions.TX_DATETIME
    
    # For each window size
    for window_size in windows_size_in_days:
        
        # Compute the sum of the transaction amounts and the number of transactions for the given window size
        SUM_AMOUNT_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').sum()
        NB_TX_WINDOW=customer_transactions['TX_AMOUNT'].rolling(str(window_size)+'d').count()
    
        # Compute the average transaction amount for the given window size
        # NB_TX_WINDOW is always >0 since current transaction is always included
        AVG_AMOUNT_TX_WINDOW=SUM_AMOUNT_TX_WINDOW/NB_TX_WINDOW
    
        # Save feature values
        customer_transactions['CUSTOMER_ID_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
        customer_transactions['CUSTOMER_ID_AVG_AMOUNT_'+str(window_size)+'DAY_WINDOW']=list(AVG_AMOUNT_TX_WINDOW)
    
    # Reindex according to transaction IDs
    customer_transactions.index=customer_transactions.TRANSACTION_ID
        
    # And return the dataframe with the new features
    return customer_transactions


Vamos calcular esses agregados para o primeiro cliente.

In [11]:
spending_behaviour_customer_0=get_customer_spending_behaviour_features(transactions_df[transactions_df.CUSTOMER_ID==0])
spending_behaviour_customer_0

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
TRANSACTION_ID,,,,,,,,,,,,,,,,,
1758,1758,2018-04-01 07:19:05,0,6076,123.59,26345,0,0,0,1,0,1.0,123.590000,1.0,123.590000,1.0,123.590000
8275,8275,2018-04-01 18:00:16,0,858,77.34,64816,0,0,0,1,0,2.0,100.465000,2.0,100.465000,2.0,100.465000
8640,8640,2018-04-01 19:02:02,0,6698,46.51,68522,0,0,0,1,0,3.0,82.480000,3.0,82.480000,3.0,82.480000
12169,12169,2018-04-02 08:51:06,0,6569,54.72,118266,1,0,0,0,0,3.0,59.523333,4.0,75.540000,4.0,75.540000
15764,15764,2018-04-02 14:05:38,0,7707,63.30,137138,1,0,0,0,0,4.0,60.467500,5.0,73.092000,5.0,73.092000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1750390,1750390,2018-09-30 13:38:41,0,3096,38.23,15773921,182,0,0,1,0,5.0,64.388000,28.0,57.306429,89.0,63.097640
1750758,1750758,2018-09-30 14:10:21,0,9441,43.60,15775821,182,0,0,1,0,6.0,60.923333,29.0,56.833793,89.0,62.433933
1751039,1751039,2018-09-30 14:34:30,0,1138,69.69,15777270,182,0,0,1,0,7.0,62.175714,29.0,57.872414,90.0,62.514556


Podemos verificar que as novas características são consistentes com o perfil do cliente (ver o notebook anterior). Para o cliente 0, o valor médio era `mean_amount`=62.26, e a frequência de transação era `mean_nb_tx_per_day`=2.18. Esses valores são de fato bem correspondidos pelas características `CUSTOMER_ID_NB_TX_30DAY_WINDOW` e `CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW`, especialmente após 30 dias.

Vamos agora gerar essas características para todos os clientes. Isso é simples usando os métodos `groupby` e `apply` do Pandas.

In [12]:
%time transactions_df=transactions_df.groupby('CUSTOMER_ID').apply(lambda x: get_customer_spending_behaviour_features(x, windows_size_in_days=[1,7,30]))
transactions_df=transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)


CPU times: user 1min 2s, sys: 1.21 s, total: 1min 3s
Wall time: 1min 7s


In [13]:
transactions_df

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0,1,1,1.0,57.160000,1.0,57.160000,1.0,57.160000
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0,1,1,1.0,81.510000,1.0,81.510000,1.0,81.510000
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0,1,1,1.0,146.000000,1.0,146.000000,1.0,146.000000
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0,1,1,1.0,64.490000,1.0,64.490000,1.0,64.490000
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0,1,1,1.0,50.990000,1.0,50.990000,1.0,50.990000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754150,1754150,2018-09-30 23:56:36,161,655,54.24,15810996,182,0,0,1,0,2.0,75.280000,12.0,67.047500,72.0,69.521111
1754151,1754151,2018-09-30 23:57:38,4342,6181,1.23,15811058,182,0,0,1,0,1.0,1.230000,21.0,22.173810,93.0,24.780753
1754152,1754152,2018-09-30 23:58:21,618,1502,6.62,15811101,182,0,0,1,0,5.0,7.368000,21.0,7.400476,65.0,7.864462
1754153,1754153,2018-09-30 23:59:52,4056,3067,55.40,15811192,182,0,0,1,0,3.0,100.696667,16.0,107.052500,51.0,102.919608


## Transformações do ID do terminal

Por fim, vamos prosseguir com as transformações do ID do terminal. O principal objetivo será extrair uma *pontuação de risco* que avalia a exposição de um determinado ID de terminal a transações fraudulentas. A pontuação de risco será definida como o número médio de transações fraudulentas que ocorreram em um ID de terminal ao longo de uma janela temporal. Assim como nas transformações do ID do cliente, usaremos três tamanhos de janela: 1, 7 e 30 dias.

Ao contrário das transformações do ID do cliente, as janelas temporais não precederão diretamente uma determinada transação. Em vez disso, elas serão deslocadas para trás por um *período de atraso*. O período de atraso leva em conta o fato de que, na prática, as transações fraudulentas só são descobertas após uma investigação de fraude ou uma reclamação do cliente. Portanto, os rótulos fraudulentos, necessários para calcular a pontuação de risco, só estão disponíveis após esse período de atraso. Como primeira aproximação, esse período de atraso será definido como uma semana. As motivações para o período de atraso serão discutidas com mais detalhes no [Capítulo 5, Estratégias de validação](Validation_Strategies).

Vamos realizar o cálculo das pontuações de risco definindo uma função `get_count_risk_rolling_window`. A função recebe como entradas o DataFrame de transações para um determinado ID de terminal, o período de atraso e uma lista de tamanhos de janela. Na primeira etapa, o número de transações e de transações fraudulentas é calculado para o período de atraso (`NB_TX_DELAY` e `NB_FRAUD_DELAY`). Na segunda etapa, o número de transações e de transações fraudulentas é calculado para cada tamanho de janela mais o período de atraso (`NB_TX_DELAY_WINDOW` e `NB_FRAUD_DELAY_WINDOW`). O número de transações e de transações fraudulentas que ocorreram para um determinado tamanho de janela, deslocado para trás pelo período de atraso, é então obtido simplesmente calculando as diferenças das quantidades obtidas para o período de atraso e o tamanho de janela mais o período de atraso:

```
NB_FRAUD_WINDOW=NB_FRAUD_DELAY_WINDOW-NB_FRAUD_DELAY
NB_TX_WINDOW=NB_TX_DELAY_WINDOW-NB_TX_DELAY
```

A pontuação de risco é finalmente obtida calculando a proporção de transações fraudulentas para cada tamanho de janela (ou 0 se nenhuma transação ocorreu para a janela fornecida):

```
RISK_WINDOW=NB_FRAUD_WINDOW/NB_TX_WINDOW
```

Além da pontuação de risco, a função também retorna o número de transações para cada tamanho de janela. Isso resulta na adição de seis novas características: o risco e o número de transações, para três tamanhos de janela.

In [14]:
def get_count_risk_rolling_window(terminal_transactions, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"):
    
    terminal_transactions=terminal_transactions.sort_values('TX_DATETIME')
    
    terminal_transactions.index=terminal_transactions.TX_DATETIME
    
    NB_FRAUD_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').sum()
    NB_TX_DELAY=terminal_transactions['TX_FRAUD'].rolling(str(delay_period)+'d').count()
    
    for window_size in windows_size_in_days:
    
        NB_FRAUD_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').sum()
        NB_TX_DELAY_WINDOW=terminal_transactions['TX_FRAUD'].rolling(str(delay_period+window_size)+'d').count()
    
        NB_FRAUD_WINDOW=NB_FRAUD_DELAY_WINDOW-NB_FRAUD_DELAY
        NB_TX_WINDOW=NB_TX_DELAY_WINDOW-NB_TX_DELAY
    
        RISK_WINDOW=NB_FRAUD_WINDOW/NB_TX_WINDOW
        
        terminal_transactions[feature+'_NB_TX_'+str(window_size)+'DAY_WINDOW']=list(NB_TX_WINDOW)
        terminal_transactions[feature+'_RISK_'+str(window_size)+'DAY_WINDOW']=list(RISK_WINDOW)
        
    terminal_transactions.index=terminal_transactions.TRANSACTION_ID
    
    # Replace NA values with 0 (all undefined risk scores where NB_TX_WINDOW is 0) 
    terminal_transactions.fillna(0,inplace=True)
    
    return terminal_transactions


In [15]:
transactions_df[transactions_df.TX_FRAUD==1]

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,TX_DURING_NIGHT,CUSTOMER_ID_NB_TX_1DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_1DAY_WINDOW,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW
3527,3527,2018-04-01 10:17:43,3774,3059,225.41,37063,0,1,1,1,0,3.0,158.073333,3.0,158.073333,3.0,158.073333
5789,5790,2018-04-01 13:31:48,4944,6050,222.26,48708,0,1,1,1,0,2.0,127.605000,2.0,127.605000,2.0,127.605000
6549,6549,2018-04-01 14:42:02,4625,9102,226.40,52922,0,1,1,1,0,4.0,167.165000,4.0,167.165000,4.0,167.165000
9583,9583,2018-04-02 01:01:05,3814,6893,59.15,90065,1,1,3,0,1,6.0,29.138333,6.0,29.138333,6.0,29.138333
10356,10355,2018-04-02 05:03:35,2513,1143,222.04,104615,1,1,1,0,1,5.0,123.740000,5.0,123.740000,5.0,123.740000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1753524,1753524,2018-09-30 19:51:48,1671,3192,128.60,15796308,182,1,3,1,0,6.0,138.358333,25.0,106.957200,82.0,75.621341
1753600,1753600,2018-09-30 20:09:00,4166,632,17.39,15797340,182,1,2,1,0,3.0,19.766667,19.0,15.984737,86.0,15.846512
1753673,1753673,2018-09-30 20:30:52,4097,1558,24.04,15798652,182,1,2,1,0,3.0,23.050000,16.0,40.440625,63.0,41.877460
1754014,1754014,2018-09-30 22:27:04,100,8604,73.85,15805624,182,1,3,1,0,2.0,48.010000,26.0,30.384231,103.0,23.627184


Vamos calcular essas seis características para o primeiro ID de terminal que contém pelo menos uma fraude:

In [16]:
# Get the first terminal ID that contains frauds
transactions_df[transactions_df.TX_FRAUD==0].TERMINAL_ID[0]

3156

In [17]:
get_count_risk_rolling_window(transactions_df[transactions_df.TERMINAL_ID==3059], delay_period=7, windows_size_in_days=[1,7,30])

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,...,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW,TERMINAL_ID_NB_TX_1DAY_WINDOW,TERMINAL_ID_RISK_1DAY_WINDOW,TERMINAL_ID_NB_TX_7DAY_WINDOW,TERMINAL_ID_RISK_7DAY_WINDOW,TERMINAL_ID_NB_TX_30DAY_WINDOW,TERMINAL_ID_RISK_30DAY_WINDOW
TRANSACTION_ID,,,,,,,,,,,,,,,,,,,,,
3527,3527,2018-04-01 10:17:43,3774,3059,225.41,37063,0,1,1,1,...,3.0,158.073333,3.0,158.073333,0.0,0.0,0.0,0.0,0.0,0.0
4732,4732,2018-04-01 11:59:14,55,3059,36.28,43154,0,0,0,1,...,2.0,35.670000,2.0,35.670000,0.0,0.0,0.0,0.0,0.0,0.0
16216,16216,2018-04-02 14:47:34,4879,3059,105.00,139654,1,0,0,0,...,10.0,76.010000,10.0,76.010000,0.0,0.0,0.0,0.0,0.0,0.0
18249,18249,2018-04-02 19:08:10,2263,3059,90.89,155290,1,0,0,0,...,7.0,50.458571,7.0,50.458571,0.0,0.0,0.0,0.0,0.0,0.0
26512,26512,2018-04-03 15:44:49,4879,3059,58.51,229489,2,0,0,0,...,14.0,71.070000,14.0,71.070000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1697944,1697944,2018-09-25 05:32:56,402,3059,57.30,15312776,177,0,0,0,...,14.0,65.167857,46.0,68.163261,1.0,0.0,9.0,0.0,36.0,0.0
1701971,1701971,2018-09-25 12:30:54,1035,3059,7.56,15337854,177,0,0,0,...,23.0,7.052174,107.0,6.763738,2.0,0.0,10.0,0.0,36.0,0.0
1704512,1704512,2018-09-25 16:37:41,1519,3059,35.79,15352661,177,0,0,0,...,7.0,41.404286,30.0,46.780000,1.0,0.0,9.0,0.0,36.0,0.0


Podemos verificar que a primeira fraude ocorreu em 2018/09/10, e que as pontuações de risco só começam a ser contadas com um atraso de uma semana.

Vamos finalmente gerar essas características para todos os terminais. Isso é simples usando os métodos `groupby` e `apply` do Pandas.

In [18]:
%time transactions_df=transactions_df.groupby('TERMINAL_ID').apply(lambda x: get_count_risk_rolling_window(x, delay_period=7, windows_size_in_days=[1,7,30], feature="TERMINAL_ID"))
transactions_df=transactions_df.sort_values('TX_DATETIME').reset_index(drop=True)


CPU times: user 2min 27s, sys: 2.23 s, total: 2min 29s
Wall time: 2min 41s


In [19]:
transactions_df

,TRANSACTION_ID,TX_DATETIME,CUSTOMER_ID,TERMINAL_ID,TX_AMOUNT,TX_TIME_SECONDS,TX_TIME_DAYS,TX_FRAUD,TX_FRAUD_SCENARIO,TX_DURING_WEEKEND,...,CUSTOMER_ID_NB_TX_7DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_7DAY_WINDOW,CUSTOMER_ID_NB_TX_30DAY_WINDOW,CUSTOMER_ID_AVG_AMOUNT_30DAY_WINDOW,TERMINAL_ID_NB_TX_1DAY_WINDOW,TERMINAL_ID_RISK_1DAY_WINDOW,TERMINAL_ID_NB_TX_7DAY_WINDOW,TERMINAL_ID_RISK_7DAY_WINDOW,TERMINAL_ID_NB_TX_30DAY_WINDOW,TERMINAL_ID_RISK_30DAY_WINDOW
0,0,2018-04-01 00:00:31,596,3156,57.16,31,0,0,0,1,...,1.0,57.160000,1.0,57.160000,0.0,0.0,0.0,0.0,0.0,0.00000
1,1,2018-04-01 00:02:10,4961,3412,81.51,130,0,0,0,1,...,1.0,81.510000,1.0,81.510000,0.0,0.0,0.0,0.0,0.0,0.00000
2,2,2018-04-01 00:07:56,2,1365,146.00,476,0,0,0,1,...,1.0,146.000000,1.0,146.000000,0.0,0.0,0.0,0.0,0.0,0.00000
3,3,2018-04-01 00:09:29,4128,8737,64.49,569,0,0,0,1,...,1.0,64.490000,1.0,64.490000,0.0,0.0,0.0,0.0,0.0,0.00000
4,4,2018-04-01 00:10:34,927,9906,50.99,634,0,0,0,1,...,1.0,50.990000,1.0,50.990000,0.0,0.0,0.0,0.0,0.0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1754150,1754150,2018-09-30 23:56:36,161,655,54.24,15810996,182,0,0,1,...,12.0,67.047500,72.0,69.521111,1.0,0.0,4.0,0.0,28.0,0.00000
1754151,1754151,2018-09-30 23:57:38,4342,6181,1.23,15811058,182,0,0,1,...,21.0,22.173810,93.0,24.780753,1.0,0.0,9.0,0.0,39.0,0.00000
1754152,1754152,2018-09-30 23:58:21,618,1502,6.62,15811101,182,0,0,1,...,21.0,7.400476,65.0,7.864462,1.0,0.0,5.0,0.0,33.0,0.00000
1754153,1754153,2018-09-30 23:59:52,4056,3067,55.40,15811192,182,0,0,1,...,16.0,107.052500,51.0,102.919608,1.0,0.0,6.0,0.0,28.0,0.00000


## Salvamento do conjunto de dados

Vamos finalmente salvar o conjunto de dados, dividido em lotes diários, usando o formato pickle.

In [22]:
DIR_OUTPUT = "./simulated-data-transformed/"

if not os.path.exists(DIR_OUTPUT):
    os.makedirs(DIR_OUTPUT)

start_date = datetime.datetime.strptime("2018-04-01", "%Y-%m-%d")

for day in range(transactions_df.TX_TIME_DAYS.max()+1):
    
    transactions_day = transactions_df[transactions_df.TX_TIME_DAYS==day].sort_values('TX_TIME_SECONDS')
    
    date = start_date + datetime.timedelta(days=day)
    filename_output = date.strftime("%Y-%m-%d")+'.pkl'
    
    # Protocol=4 required for Google Colab
    transactions_day.to_pickle(DIR_OUTPUT+filename_output, protocol=4)

O conjunto de dados gerado também está disponível no Github em `https://github.com/Fraud-Detection-Handbook/simulated-data-transformed`.